# Протоколы взаимодействия, MCP

**Цель:** разобрать протокол MCP (Model Context Protocol): сервер предоставляет инструменты (tools), клиент (агент или пользователь) вызывает их по имени с аргументами.

**Используемые примеры:**
- `topic_2_agents/example_4_mcp_route_agents` — MCP-сервер на FastAPI-MCP (маршруты); агент, вызывающий MCP через REST.

In [ ]:
%autosave 60

## Настройка окружения

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
%cd ..

## Теория: MCP

**Model Context Protocol (MCP)** — способ, которым приложение (сервер) предоставляет модели контекст и инструменты: описание тулов (имя, параметры), вызов по имени с JSON-аргументами. Транспорт может быть stdio, HTTP и др.

В примере: FastAPI-приложение с эндпоинтом `/route` (расчёт маршрута) оборачивается через **FastAPI-MCP** в MCP-сервер; агент на стороне клиента извлекает из запроса пользователя параметры (origin, destination, mode) через LLM и вызывает REST `/route`.

### Запуск MCP-сервера (server.py)

Сервер нужно запустить в отдельном терминале: `uvicorn server:app --host 0.0.0.0 --port 8000` из директории `ai_agents_course/topic_2_agents/example_4_mcp_route_agents`. Ниже — команда для запуска в фоне (или выполните вручную в другом окне).

In [ ]:
# Запуск сервера в фоне (остановите ядро или процесс вручную после демо)
import subprocess
import time

server_dir = ROOT / "ai_agents_course" / "topic_2_agents" / "example_4_mcp_route_agents"
proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(server_dir),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)
print("Сервер запущен на http://127.0.0.1:8000. Для остановки: proc.terminate()")

### Агент: запрос → LLM извлекает параметры → вызов /route (query_route_ollama.py)

Текстовый запрос пользователя передаётся в Ollama; модель возвращает JSON (origin, destination, mode). По этим параметрам вызывается REST-сервис маршрутов.

In [ ]:
%cd ai_agents_course/topic_2_agents/example_4_mcp_route_agents
!python query_route_ollama.py "От Красной площади до Шереметьево на машине"
%cd ../../..

## Итоги

- MCP задаёт контракт между сервером (инструменты) и клиентом (агент/пользователь).
- В примере агент использует LLM для извлечения структурированных параметров из текста и вызывает внешний сервис (REST), что типично для интеграции агентов с MCP-серверами.